# Forex Trading Strategy Analysis
## EUR/USD Backtesting Results Analyzer

This notebook analyzes EUR/USD forex trading data to identify profitable trading strategies based on various technical indicators and market conditions.

In [36]:
# Import required libraries
import pandas as pd
import numpy as np
from IPython.display import display, HTML

# ================== DATA LOADING & PREPROCESSING ==================

def load_and_clean_data(filepath='./eurusd.csv'):
    """
    Load EUR/USD data from CSV and clean NaN values.
    
    Args:
        filepath (str): Path to the CSV file containing trading data
    
    Returns:
        pd.DataFrame: Cleaned dataframe with trading data
    """
    df = pd.read_csv(filepath)
    
    # Define columns that should have NaN replaced with 0
    columns_to_fillna = [
        'SL', 'TP', 'SL 5M CC', 'SL 5M Stop', 
        'SL Breakout', 'Hours Until News', 'Extra'
    ]
    
    # Fill NaN values to prevent calculation errors
    for col in columns_to_fillna:
        if col in df.columns:
            df[col] = df[col].fillna(0)
    
    return df

def percentage(value, total):
    """
    Calculate percentage of value over total.
    
    Args:
        value (int/float): The numerator value
        total (int/float): The denominator value
    
    Returns:
        str: Formatted percentage string (e.g., "45.5%")
    """
    if total == 0:
        return "0.0%"
    return f"{value / total * 100:.1f}%"

# Load the data
df = load_and_clean_data()

# ================== PROFITABLE TRADES ANALYSIS ==================

def calculate_profitable_trades(df):
    """
    Calculate profitability statistics for different trading strategies.
    
    Args:
        df (pd.DataFrame): Trading data
    
    Returns:
        pd.DataFrame: Profitability statistics for various strategies
    """
    # Define trading strategies with their filters
    strategies = {
        'Total': lambda d: d[(d['SL'] != d['Pullback'])],
        'With Extra': lambda d: d[((d['SL'] != d['Pullback']) | 
                                  ((d['SL'] == d['Pullback']) & (d['Extra'].notna())))],
        'With EMA': lambda d: d[(d['SL'] != d['Pullback']) & (d['EMA'] == d['Direction'])],
        'Against EMA': lambda d: d[(d['SL'] != d['Pullback']) & (d['EMA'] != d['Direction'])],
        'BOS Trades': lambda d: d[(d['SL'] != d['Pullback']) & (d['BOS/CH'] == 'BOS')],
        'CH Trades': lambda d: d[(d['SL'] != d['Pullback']) & (d['BOS/CH'] == 'CH')],
        'With EMA & BOS': lambda d: d[(d['SL'] != d['Pullback']) & 
                                      (d['EMA'] == d['Direction']) & (d['BOS/CH'] == 'BOS')],
        'With EMA & CH': lambda d: d[(d['SL'] != d['Pullback']) & 
                                     (d['EMA'] == d['Direction']) & (d['BOS/CH'] == 'CH')],
    }
    
    results = {'Data': list(strategies.keys())}
    
    # Calculate for each Risk-Reward Ratio
    for rrr in [1, 2, 3]:
        rrr_results = []
        for strategy_name, filter_func in strategies.items():
            filtered = filter_func(df)
            
            if strategy_name == 'With Extra':
                # Special handling for 'With Extra' strategy
                profitable = filtered[filtered['TP'] >= (filtered['SL'] + filtered['Extra'].fillna(0)) * rrr]
            else:
                profitable = filtered[filtered['TP'] >= filtered['SL'] * rrr]
            
            rrr_results.append(percentage(len(profitable), len(df)))
        
        results[f'1:{rrr} RRR'] = rrr_results
    
    # Create DataFrame and sort by best 1:1 RRR performance
    df_results = pd.DataFrame(results)
    df_results['Value_num'] = df_results['1:1 RRR'].str.rstrip('%').astype(float)
    return df_results.sort_values('Value_num', ascending=False).drop(columns='Value_num').reset_index(drop=True)

# Display profitable trades analysis
display(HTML("<h2>Profitable Trades</h2><p>What are simple trading ideas that are profitable?</p>"))
df_profitable = calculate_profitable_trades(df)
display(df_profitable)

,Data,1:1 RRR,1:2 RRR,1:3 RRR
0,With Extra,51.6%,32.3%,29.0%
1,Total,38.7%,25.8%,22.6%
2,With EMA,25.8%,12.9%,9.7%
3,BOS Trades,25.8%,16.1%,16.1%
4,With EMA & BOS,19.4%,9.7%,9.7%
5,Against EMA,12.9%,12.9%,12.9%
6,CH Trades,12.9%,9.7%,6.5%
7,With EMA & CH,6.5%,3.2%,0.0%


In [37]:
# ================== ENTRY TIMING ANALYSIS ==================

def analyze_entry_timing(df):
    """
    Analyze different entry timing strategies and their success rates.
    
    This function compares various entry methods:
    - 1M Confirmation Candle: Entry on 1-minute candle confirmation
    - 5M Confirmation Candle: Entry on 5-minute candle confirmation  
    - 5M Stop: Entry with 5-minute stop loss level
    - 5M Breakout: Entry on 5-minute breakout signal
    
    Args:
        df (pd.DataFrame): Trading data with entry signals
    
    Returns:
        pd.DataFrame: Entry timing statistics sorted by success rate
    """
    # Calculate winning trades for each entry method
    entry_methods = {
        '1M Confirmation Candle': {
            'filter': lambda d: d['SL'] != 0,
            'profitable': lambda d: d[(d['SL'] != d['Pullback']) & (d['TP'] > d['SL'])],
            'sl_col': 'SL'
        },
        '5M Confirmation Candle': {
            'filter': lambda d: d['SL 5M CC'] != 0,
            'profitable': lambda d: d[(d['SL'] != d['Pullback']) & (d['TP'] > d['SL 5M CC'])],
            'sl_col': 'SL 5M CC'
        },
        '5M Stop': {
            'filter': lambda d: d['SL 5M Stop'] != 0,
            'profitable': lambda d: d[(d['SL'] != d['Pullback']) & (d['TP'] > d['SL 5M Stop'])],
            'sl_col': 'SL 5M Stop'
        },
        '5M Breakout': {
            'filter': lambda d: d['SL Breakout'] != 0,
            'profitable': lambda d: d[(d['SL'] != d['Pullback']) & (d['TP'] > d['SL Breakout'])],
            'sl_col': 'SL Breakout'
        }
    }
    
    results = []
    for method_name, method_config in entry_methods.items():
        # Get relevant trades for this method
        relevant_trades = df[method_config['filter'](df)]
        profitable_trades = method_config['profitable'](df)
        total_trades = len(relevant_trades)
        wins = len(profitable_trades)
        losses = total_trades - wins
        
        # Calculate metrics for different scenarios
        sl_col = method_config['sl_col']
        
        # With Extra calculation
        with_extra_filter = ((df['SL'] != df['Pullback']) | (df['Extra'] != 0))
        with_extra_profitable = df[with_extra_filter & (df['TP'] > df[sl_col])]
        
        # 1:3 RRR with Extra
        rrr3_with_extra = df[with_extra_filter & (df['TP'] > df[sl_col] * 3)]
        
        results.append({
            'Idea': method_name,
            'Count': f"{wins}W - {losses}L",
            'Percentage': percentage(wins, total_trades),
            'With Extra': percentage(len(with_extra_profitable), total_trades),
            'Extra & 1:3 RRR': percentage(len(rrr3_with_extra), total_trades)
        })
    
    # Convert to DataFrame and sort by win percentage
    df_results = pd.DataFrame(results)
    df_results['Value_num'] = df_results['Percentage'].str.rstrip('%').astype(float)
    return df_results.sort_values('Value_num', ascending=False).drop(columns='Value_num').reset_index(drop=True)

# Display entry timing analysis
display(HTML("<h2>When To Enter</h2><p>Following market structure, price taps order block. This is when the signal is received. Now - when to enter the trade?</p>"))
df_entry_timing = analyze_entry_timing(df)
display(df_entry_timing)

,Idea,Count,Percentage,With Extra,Extra & 1:3 RRR
0,5M Breakout,8W - 11L,42.1%,57.9%,26.3%
1,5M Stop,9W - 13L,40.9%,54.5%,27.3%
2,1M Confirmation Candle,12W - 19L,38.7%,48.4%,29.0%
3,5M Confirmation Candle,8W - 17L,32.0%,44.0%,24.0%


In [38]:
# ================== BOS (BREAK OF STRUCTURE) ANALYSIS ==================

def analyze_bos_combinations(df):
    """
    Analyze Break of Structure (BOS) trades combined with other indicators.
    
    BOS is a key market structure concept where price breaks previous swing highs/lows,
    potentially indicating trend continuation or reversal.
    
    Args:
        df (pd.DataFrame): Trading data with BOS/CH indicators
    
    Returns:
        pd.DataFrame: BOS combination statistics sorted by performance
    """
    # Define BOS combination strategies
    bos_strategies = {
        'BOS Trades': lambda d: d[(d['BOS/CH'] == 'BOS')],
        'BOS + Buy': lambda d: d[(d['BOS/CH'] == 'BOS') & (d['Direction'] == 'Buy')],
        'BOS + Sell': lambda d: d[(d['BOS/CH'] == 'BOS') & (d['Direction'] == 'Sell')],
        'BOS + EMA': lambda d: d[(d['BOS/CH'] == 'BOS') & (d['EMA'] == d['Direction'])],
        'BOS + Against EMA': lambda d: d[(d['BOS/CH'] == 'BOS') & (d['EMA'] != d['Direction'])],
        'BOS + HTF Buy': lambda d: d[(d['BOS/CH'] == 'BOS') & 
                                     (d['30M Leg'].isin(['Above H', 'Above L']))],
        'BOS + HTF Sell': lambda d: d[(d['BOS/CH'] == 'BOS') & 
                                      (d['30M Leg'].isin(['Below H', 'Below L']))],
        'BOS + News': lambda d: d[(d['BOS/CH'] == 'BOS') & (~d['News Event'].isna())],
        'BOS + Without News': lambda d: d[(d['BOS/CH'] == 'BOS') & (d['News Event'].isna())]
    }
    
    results = {'Idea': list(bos_strategies.keys())}
    
    # Calculate performance for each RRR level
    for rrr in [1, 2, 3]:
        rrr_results = []
        for strategy_name, filter_func in bos_strategies.items():
            # Apply base filter (valid trades) and strategy filter
            valid_trades = df[df['SL'] != df['Pullback']]
            strategy_trades = filter_func(valid_trades)
            profitable = strategy_trades[strategy_trades['TP'] >= strategy_trades['SL'] * rrr]
            
            rrr_results.append(percentage(len(profitable), len(df)))
        
        results[f'1:{rrr} RRR'] = rrr_results
    
    # Create DataFrame and sort by best 1:1 RRR performance
    df_results = pd.DataFrame(results)
    df_results['Value_num'] = df_results['1:1 RRR'].str.rstrip('%').astype(float)
    return df_results.sort_values('Value_num', ascending=False).drop(columns='Value_num').reset_index(drop=True)

# Display BOS breakdown analysis
display(HTML("<h2>BOS Breakdown</h2><p>If BOS trades have highest chances, what about BOS + something else?</p>"))
df_bos = analyze_bos_combinations(df)
display(df_bos)

,Idea,1:1 RRR,1:2 RRR,1:3 RRR
0,BOS Trades,25.8%,16.1%,16.1%
1,BOS + EMA,19.4%,9.7%,9.7%
2,BOS + HTF Sell,19.4%,12.9%,12.9%
3,BOS + Without News,16.1%,12.9%,12.9%
4,BOS + Buy,12.9%,6.5%,6.5%
5,BOS + Sell,12.9%,9.7%,9.7%
6,BOS + News,9.7%,3.2%,3.2%
7,BOS + Against EMA,6.5%,6.5%,6.5%
8,BOS + HTF Buy,6.5%,3.2%,3.2%


In [39]:
# ================== STRATEGY FRAMEWORK ==================

def calculate_rrr_stats(data_df, strategy_name):
    """
    Calculate comprehensive Risk-Reward Ratio statistics for a trading strategy.
    
    This function evaluates a strategy's performance across different RRR targets
    and calculates key metrics including win rate, edge over breakeven, and
    expected outcome in R-multiples.
    
    Args:
        data_df (pd.DataFrame): Filtered DataFrame containing trades for this strategy
        strategy_name (str): Name of the strategy for labeling
    
    Returns:
        pd.DataFrame: Statistics table with the following metrics:
            - Total trades: Number of trades in the strategy
            - Wins/Losses: Count of profitable vs unprofitable trades
            - Win Rate: Percentage of winning trades
            - Edge: Win rate minus breakeven rate for the RRR
            - Outcome: Net result in R-multiples (profit factor)
    """
    total_trades = len(data_df)
    
    # RRR configurations with their breakeven win rates
    # For 1:1 RRR, you need 50% wins to break even
    # For 1:2 RRR, you need 33.3% wins to break even
    # For 1:3 RRR, you need 25% wins to break even
    rrr_configs = [
        (1, 50.0),   # 1:1 RRR
        (2, 33.3),   # 1:2 RRR
        (3, 25.0),   # 1:3 RRR
    ]
    
    # Handle empty strategy results
    if total_trades == 0:
        summary_data = {
            strategy_name: ['Total trades', 'Wins', 'Losses', 'Win Rate', 'Edge', 'Outcome']
        }
        for ratio, _ in rrr_configs:
            summary_data[f'1:{ratio} RRR'] = [0, 0, 0, '0.0%', '0.0%', '0R']
        return pd.DataFrame(summary_data)
    
    # Calculate statistics for each RRR level
    summary_data = {
        strategy_name: ['Total trades', 'Wins', 'Losses', 'Win Rate', 'Edge', 'Outcome']
    }
    
    for ratio, breakeven_rate in rrr_configs:
        # Find profitable trades for this RRR
        profitable = data_df[data_df['TP'] > ratio * data_df['SL']]
        wins = len(profitable)
        losses = total_trades - wins
        win_rate = (wins / total_trades) * 100
        
        # Edge is how much better we perform than breakeven
        edge = win_rate - breakeven_rate
        
        # Calculate expected outcome in R-multiples
        # Winners make 'ratio' R, losers lose 1R
        outcome = (wins * ratio) - losses
        
        summary_data[f'1:{ratio} RRR'] = [
            total_trades,
            wins,
            losses,
            f"{win_rate:.1f}%",
            f"{edge:.1f}%",
            f"{outcome}R"
        ]
    
    return pd.DataFrame(summary_data)


class Strategy:
    """
    Encapsulates a trading strategy with its filter logic and metadata.
    
    Attributes:
        name (str): Strategy identifier
        filter_func (callable): Function that filters trades based on strategy rules
        description (str): Human-readable description of the strategy
    """
    
    def __init__(self, name, filter_func, description=""):
        """
        Initialize a trading strategy.
        
        Args:
            name (str): Strategy name
            filter_func (callable): Lambda or function that takes df and returns filtered df
            description (str): Optional description of the strategy
        """
        self.name = name
        self.filter_func = filter_func
        self.description = description
    
    def apply(self, df):
        """
        Apply the strategy filter to a dataframe of trades.
        
        Args:
            df (pd.DataFrame): Trading data
            
        Returns:
            pd.DataFrame: Filtered trades matching strategy criteria
        """
        return self.filter_func(df)

In [40]:
# ================== STRATEGY DEFINITIONS ==================

# Initialize base strategy list
strategies = [
    Strategy(
        "Plain Strategy",
        lambda df: df,
        "Baseline: All trades without any filtering"
    )
]

In [41]:
# ================== STRATEGY FACTORY ==================

def create_strategy_library():
    """
    Create a comprehensive library of trading strategies for backtesting.
    
    This function generates strategies across multiple categories:
    1. Technical Indicators (EMA, BOS/CH)
    2. Risk Management (Stop Loss levels)
    3. Market Structure (Trend alignment)
    4. Time-based (Weekdays, News events)
    5. Combined filters (Multi-factor strategies)
    
    Returns:
        list: List of Strategy objects ready for backtesting
    """
    strategy_configs = []
    
    # ========== TECHNICAL INDICATOR STRATEGIES ==========
    # EMA (Exponential Moving Average) based strategies
    strategy_configs.extend([
        ("EMA + Trade Direction", 
         lambda df: df[df['EMA'] == df['Direction']], 
         "Trade with EMA trend"),
        ("EMA + Opposite Trade Direction", 
         lambda df: df[df['EMA'] != df['Direction']], 
         "Counter-trend trades"),
        ("EMA + BOS", 
         lambda df: df[(df['EMA'] == df['Direction']) & (df['BOS/CH'] == 'BOS')], 
         "Trend + Break of Structure"),
        ("EMA + CH", 
         lambda df: df[(df['EMA'] == df['Direction']) & (df['BOS/CH'] == 'CH')], 
         "Trend + Change of Character"),
    ])
    
    # Market Structure strategies
    strategy_configs.extend([
        ("BOS", 
         lambda df: df[df['BOS/CH'] == 'BOS'], 
         "Break of Structure trades only"),
        ("CH", 
         lambda df: df[df['BOS/CH'] == 'CH'], 
         "Change of Character trades only"),
    ])
    
    # ========== RISK MANAGEMENT STRATEGIES ==========
    # Stop Loss size filtering
    strategy_configs.extend([
        ("Conservative: SL <= 2 pips", 
         lambda df: df[df['SL'] <= 2], 
         "Very tight stop losses"),
        ("Moderate Risk: SL 3-6 pips", 
         lambda df: df[(df['SL'] >= 3) & (df['SL'] <= 6)], 
         "Medium stop losses"),
        ("Aggressive: SL >= 7 pips", 
         lambda df: df[df['SL'] >= 7], 
         "Wide stop losses"),
    ])
    
    # Risk-adjusted market structure combinations
    strategy_configs.extend([
        ("BOS + Conservative SL", 
         lambda df: df[(df['BOS/CH'] == 'BOS') & (df['SL'] <= 2)], 
         "BOS with tight stops"),
        ("BOS + Moderate SL", 
         lambda df: df[(df['BOS/CH'] == 'BOS') & (df['SL'] >= 3) & (df['SL'] <= 6)], 
         "BOS with medium stops"),
    ])
    
    # ========== TIME-BASED STRATEGIES ==========
    # Day of week analysis
    weekdays = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']
    for day in weekdays:
        strategy_configs.append(
            (f"{day} Only", 
             lambda df, d=day: df[df['Weekday'] == d], 
             f"Trades on {day}")
        )
    
    # ========== HIGHER TIMEFRAME TREND ==========
    # 30-minute timeframe trend alignment
    strategy_configs.extend([
        ("With 30M Trend", 
         lambda df: df[(df['30M Leg'].isin(['Above H', 'Above L']) & (df['Direction'] == 'Buy')) | (df['30M Leg'].isin(['Below H', 'Below L']) & (df['Direction'] == 'Sell'))], 
         "Higher timeframe uptrend"),
    ])
    
    # ========== NEWS EVENT STRATEGIES ==========
    strategy_configs.extend([
        ("No News Events", 
         lambda df: df[df['News Event'].isna()], 
         "Avoid news volatility"),
        ("News Event Present", 
         lambda df: df[~df['News Event'].isna()], 
         "Trade during news"),
        ("News in 2+ Hours", 
         lambda df: df[(~df['News Event'].isna()) & (df['Hours Until News'] >= 2)], 
         "Trade before news impact"),
    ])
    
    # ========== COMBINED MULTI-FACTOR STRATEGIES ==========
    # These combine multiple indicators for potentially higher probability setups
    strategy_configs.extend([
        ("EMA + BOS + Low Risk", 
         lambda df: df[(df['EMA'] == df['Direction']) & (df['BOS/CH'] == 'BOS') & (df['SL'] <= 3)], 
         "Triple confirmation setup"),
        ("BOS + Bullish 30M + No News", 
         lambda df: df[(df['BOS/CH'] == 'BOS') & 
                      (df['30M Leg'].isin(['Above H', 'Above L'])) & 
                      (df['News Event'].isna())], 
         "Clean trend continuation"),
        ("Strong Alignment + EMA", 
         lambda df: df[((df['30M Leg'] == 'Above H') & (df['Direction'] == 'Buy') & 
                       (df['EMA'] == 'Buy')) | 
                      ((df['30M Leg'] == 'Below L') & (df['Direction'] == 'Sell') & 
                       (df['EMA'] == 'Sell'))], 
         "Full trend confluence"),
    ])
    
    # Convert configurations to Strategy objects
    return [Strategy(name, func, desc) for name, func, desc in strategy_configs]

# Add all strategies to the main list
strategies.extend(create_strategy_library())

# ================== STRATEGY EVALUATION ==================

def evaluate_all_strategies(df, strategies):
    """
    Run backtesting on all strategies and compile results.
    
    Args:
        df (pd.DataFrame): Trading data
        strategies (list): List of Strategy objects
        
    Returns:
        dict: Dictionary mapping strategy names to their performance DataFrames
    """
    strategy_results = {}
    
    for strategy in strategies:
        # Apply strategy filter
        filtered_df = strategy.apply(df)
        
        # Calculate RRR statistics
        summary_df = calculate_rrr_stats(filtered_df, strategy.name)
        
        # Store results
        strategy_results[strategy.name] = summary_df
    
    return strategy_results

def get_top_strategies(strategy_results, rrr_column, top_n=5):
    """
    Extract top performing strategies for a specific RRR.
    
    Args:
        strategy_results (dict): Dictionary of strategy results
        rrr_column (str): Column name for RRR (e.g., '1:2 RRR')
        top_n (int): Number of top strategies to return
        
    Returns:
        pd.DataFrame: Top strategies ranked by outcome
    """
    strategy_performance = []
    
    for strategy_name, df in strategy_results.items():
        # Extract performance metrics
        total_trades = df[rrr_column].iloc[0]
        wins = df[rrr_column].iloc[1]
        losses = df[rrr_column].iloc[2]
        win_rate = df[rrr_column].iloc[3]
        edge = df[rrr_column].iloc[4]
        outcome_str = df[rrr_column].iloc[5]
        
        # Parse outcome value for sorting
        outcome = int(outcome_str.replace('R', ''))
        
        strategy_performance.append({
            'Strategy': strategy_name,
            'Trades': total_trades,
            'Wins': wins,
            'Losses': losses,
            'Win Rate': win_rate,
            'Edge': edge,
            'Outcome': outcome_str,
            'outcome_value': outcome
        })
    
    # Sort by outcome and get top N
    top_strategies = sorted(
        strategy_performance, 
        key=lambda x: x['outcome_value'], 
        reverse=True
    )[:top_n]
    
    # Remove sorting key from display
    for strat in top_strategies:
        del strat['outcome_value']
    
    return pd.DataFrame(top_strategies)

# ================== RUN BACKTESTING ==================

# Evaluate all strategies
print(f"Evaluating {len(strategies)} strategies...")
strategy_results = evaluate_all_strategies(df, strategies)

# Display top performing strategies for each RRR
display(HTML("<h2>🏆 TOP Performing Strategies</h2>"))

rrr_configs = [
    ('1:1 RRR', '1:1'),
    ('1:2 RRR', '1:2'),
    ('1:3 RRR', '1:3'),
]

for rrr_column, rrr_label in rrr_configs:
    display(HTML(f"<h3>Best {rrr_label} Strategies</h3>"))
    
    # Get and display top 5 strategies
    top_5_df = get_top_strategies(strategy_results, rrr_column, top_n=5)
    top_5_df = top_5_df.rename(columns={'Strategy': f'Top {rrr_label} Strategies'})
    
    # Style the table for better readability
    styled_df = top_5_df.style.set_properties(
        subset=[f'Top {rrr_label} Strategies'], 
        **{'width': '300px', 'font-weight': 'bold'}
    ).set_properties(
        subset=['Outcome'],
        **{'color': 'green', 'font-weight': 'bold'}
    )
    
    display(styled_df)
    print()  # Add spacing

Evaluating 24 strategies...


,Top 1:1 Strategies,Trades,Wins,Losses,Win Rate,Edge,Outcome
0,Friday Only,5,5,0,100.0%,50.0%,5R
1,With 30M Trend,18,11,7,61.1%,11.1%,4R
2,EMA + BOS + Low Risk,6,4,2,66.7%,16.7%,2R
3,Strong Alignment + EMA,8,5,3,62.5%,12.5%,2R
4,Plain Strategy,31,16,15,51.6%,1.6%,1R


,Top 1:2 Strategies,Trades,Wins,Losses,Win Rate,Edge,Outcome
0,Friday Only,5,4,1,80.0%,46.7%,7R
1,With 30M Trend,18,8,10,44.4%,11.1%,6R
2,EMA + BOS + Low Risk,6,4,2,66.7%,33.4%,6R
3,Strong Alignment + EMA,8,4,4,50.0%,16.7%,4R
4,BOS + Conservative SL,3,2,1,66.7%,33.4%,3R


,Top 1:3 Strategies,Trades,Wins,Losses,Win Rate,Edge,Outcome
0,Friday Only,5,4,1,80.0%,55.0%,11R
1,With 30M Trend,18,7,11,38.9%,13.9%,10R
2,EMA + BOS + Low Risk,6,4,2,66.7%,41.7%,10R
3,BOS,20,7,13,35.0%,10.0%,8R
4,EMA + Opposite Trade Direction,10,4,6,40.0%,15.0%,6R


In [42]:
# ================== DETAILED STRATEGY RESULTS ==================

def display_detailed_results(strategy_results, display_all=False):
    """
    Display detailed results for all or selected strategies.
    
    Args:
        strategy_results (dict): Dictionary of strategy performance DataFrames
        display_all (bool): If True, show all strategies; if False, show only profitable ones
    """
    if display_all:
        display(HTML(f"<h2>📊 All Strategy Results ({len(strategy_results)} strategies)</h2>"))
        strategies_to_display = strategy_results.items()
    else:
        # Filter for profitable strategies (positive outcome at 1:1 RRR)
        profitable_strategies = []
        for name, df in strategy_results.items():
            outcome_str = df['1:1 RRR'].iloc[5]
            outcome = int(outcome_str.replace('R', ''))
            if outcome > 0:
                profitable_strategies.append((name, df))
        
        display(HTML(f"<h2>💰 Profitable Strategies ({len(profitable_strategies)} of {len(strategy_results)})</h2>"))
        strategies_to_display = profitable_strategies
    
    # Display each strategy's results
    for strategy_name, summary_df in strategies_to_display:
        # Style the DataFrame for better readability
        styled_df = summary_df.style.set_properties(
            subset=[strategy_name], 
            **{'width': '300px', 'font-weight': 'bold'}
        ).set_properties(
            **{'text-align': 'center'}
        )
        
        # Highlight positive outcomes in green, negative in red
        for col in ['1:1 RRR', '1:2 RRR', '1:3 RRR']:
            outcome_str = summary_df[col].iloc[5]
            outcome = int(outcome_str.replace('R', ''))
            color = 'green' if outcome > 0 else 'red' if outcome < 0 else 'gray'
            styled_df = styled_df.set_properties(
                subset=[col],
                **{'color': color if summary_df.index[5] == 'Outcome' else 'white'}
            )
        
        display(styled_df)
        print()  # Add spacing

# Display only profitable strategies by default
display_detailed_results(strategy_results, display_all=False)

# Option to see all strategies (uncomment to view)
# display_detailed_results(strategy_results, display_all=True)

,Plain Strategy,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,31,31,31
1,Wins,16,11,9
2,Losses,15,20,22
3,Win Rate,51.6%,35.5%,29.0%
4,Edge,1.6%,2.2%,4.0%
5,Outcome,1R,2R,5R


,EMA + Trade Direction,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,21,21,21
1,Wins,11,7,5
2,Losses,10,14,16
3,Win Rate,52.4%,33.3%,23.8%
4,Edge,2.4%,0.0%,-1.2%
5,Outcome,1R,0R,-1R


,EMA + CH,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,5,5,5
1,Wins,3,2,0
2,Losses,2,3,5
3,Win Rate,60.0%,40.0%,0.0%
4,Edge,10.0%,6.7%,-25.0%
5,Outcome,1R,1R,-5R


,CH,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,11,11,11
1,Wins,6,4,2
2,Losses,5,7,9
3,Win Rate,54.5%,36.4%,18.2%
4,Edge,4.5%,3.1%,-6.8%
5,Outcome,1R,1R,-3R


,Moderate Risk: SL 3-6 pips,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,15,15,15
1,Wins,8,4,2
2,Losses,7,11,13
3,Win Rate,53.3%,26.7%,13.3%
4,Edge,3.3%,-6.6%,-11.7%
5,Outcome,1R,-3R,-7R


...

,Friday Only,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,5,5,5
1,Wins,5,4,4
2,Losses,0,1,1
3,Win Rate,100.0%,80.0%,80.0%
4,Edge,50.0%,46.7%,55.0%
5,Outcome,5R,7R,11R


,With 30M Trend,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,18,18,18
1,Wins,11,8,7
2,Losses,7,10,11
3,Win Rate,61.1%,44.4%,38.9%
4,Edge,11.1%,11.1%,13.9%
5,Outcome,4R,6R,10R


,News Event Present,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,19,19,19
1,Wins,10,6,5
2,Losses,9,13,14
3,Win Rate,52.6%,31.6%,26.3%
4,Edge,2.6%,-1.7%,1.3%
5,Outcome,1R,-1R,1R


,EMA + BOS + Low Risk,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,6,6,6
1,Wins,4,4,4
2,Losses,2,2,2
3,Win Rate,66.7%,66.7%,66.7%
4,Edge,16.7%,33.4%,41.7%
5,Outcome,2R,6R,10R


,Strong Alignment + EMA,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,8,8,8
1,Wins,5,4,3
2,Losses,3,4,5
3,Win Rate,62.5%,50.0%,37.5%
4,Edge,12.5%,16.7%,12.5%
5,Outcome,2R,4R,4R
